# 실습 5-3 · 지도학습으로 탐지: 랜덤 포레스트

UNSW-NB15에는 정답이 있어 정답으로 학습하는 지도학습도 할 수 있습니다. 05b와 같은 피처 · 같은 평가셋으로 비교합니다.
1) 비지도 vs 지도  2) 공격 유형 분류  3) 피처 중요도  4) 처음 보는 공격  5) 다른 시간대

In [1]:
import os
import numpy as np
import pandas as pd

assert os.path.exists("../data/processed/train.csv"), "05a_data_preprocessing.ipynb를 먼저 실행하세요"
train = pd.read_csv("../data/processed/train.csv", dtype={"dsport": str})
test = pd.read_csv("../data/processed/test.csv", dtype={"dsport": str})
print(f"학습 {len(train)}건 (공격 {train['label'].sum()}) / 평가 {len(test)}건 (공격 {test['label'].sum()})")

학습 3520건 (공격 166) / 평가 1509건 (공격 71)


In [2]:
DESIGNED = ["log_ct_srv_dst", "log_ct_dst_ltm", "log_ct_src_dport_ltm",       # 연결 행위
            "log_sbytes_vs_svc", "log_dbytes_vs_svc", "log_rate_vs_svc",      # 규모 (서비스 평소 대비)
            "byte_ratio_vs_svc", "pkt_ratio_vs_svc"]                          # 방향 (서비스 평소 대비)
CATS = ["DoS", "Reconnaissance", "Exploits"]

In [3]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import confusion_matrix, fbeta_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

def rf():
    # class_weight="balanced": 공격이 5%뿐이라 공격 건에 가중치를 더 줌
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", min_samples_leaf=2, random_state=42)

def report(y, score, pred, cats=None):
    r = {"정밀도": precision_score(y, pred), "재현율": recall_score(y, pred), "F2": fbeta_score(y, pred, beta=2),
         "AUC-ROC": roc_auc_score(y, score), "오탐": ((pred == 1) & (y == 0)).sum(), "미탐": ((pred == 0) & (y == 1)).sum()}
    if cats is not None:
        for c in CATS:
            r[c[:5] + " 재현율"] = pred[(cats == c).values].mean()
    return r

## 1) 비지도(Isolation Forest) vs 지도(랜덤 포레스트)

In [4]:
y_tr, y_te = train["label"], test["label"]

sc = StandardScaler().fit(train[DESIGNED])
iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=42).fit(sc.transform(train[DESIGNED]))
iso_score = -iso.score_samples(sc.transform(test[DESIGNED]))
iso_pred = (iso.predict(sc.transform(test[DESIGNED])) == -1).astype(int)

model = rf().fit(train[DESIGNED], y_tr)
prob = model.predict_proba(test[DESIGNED])[:, 1]           # 공격일 확률
rf_pred = (prob >= 0.5).astype(int)

pd.DataFrame({"비지도 IF (정답 안 씀)": report(y_te, iso_score, iso_pred, test["attack_cat"]),
              "지도 RF (정답으로 학습)": report(y_te, prob, rf_pred, test["attack_cat"])}).T.round(3)

,정밀도,재현율,F2,AUC-ROC,오탐,미탐,DoS 재현율,Recon 재현율,Explo 재현율
비지도 IF (정답 안 씀),0.256,0.31,0.297,0.875,64.0,49.0,0.444,0.143,0.304
지도 RF (정답으로 학습),0.835,0.93,0.909,0.998,13.0,5.0,0.926,1.000,0.870


### 확률 임계값을 바꾸면 (지도 RF): 비지도의 contamination과 같은 역할

In [5]:
for th in [0.3, 0.5, 0.7]:
    p = (prob >= th).astype(int)
    print(f"{th}: 경보 {p.sum():>3}건, 정밀도 {precision_score(y_te, p):.2f}, 재현율 {recall_score(y_te, p):.2f}")

0.3: 경보  97건, 정밀도 0.71, 재현율 0.97
0.5: 경보  79건, 정밀도 0.84, 재현율 0.93
0.7: 경보  61건, 정밀도 0.97, 재현율 0.83


## 2) 공격 유형 4종 분류: 대각선이 맞힌 건수

In [6]:
mc = rf().fit(train[DESIGNED], train["attack_cat"])
labels = ["Normal"] + CATS
pd.DataFrame(confusion_matrix(test["attack_cat"], mc.predict(test[DESIGNED]), labels=labels),
             index=["실제 " + l for l in labels], columns=["예측 " + l for l in labels])

,예측 Normal,예측 DoS,예측 Reconnaissance,예측 Exploits
실제 Normal,1424,8,2,4
실제 DoS,1,15,3,8
실제 Reconnaissance,0,0,20,1
실제 Exploits,3,9,2,9


## 3) 피처 중요도: 모델이 무엇을 보고 판단했나

In [7]:
pd.Series(model.feature_importances_, index=DESIGNED).sort_values(ascending=False).round(3)

pkt_ratio_vs_svc        0.248
log_ct_dst_ltm          0.168
byte_ratio_vs_svc       0.136
log_dbytes_vs_svc       0.132
log_rate_vs_svc         0.110
log_ct_srv_dst          0.108
log_sbytes_vs_svc       0.090
log_ct_src_dport_ltm    0.009
dtype: float64

## 4) 처음 보는 공격: 학습에서 Exploits를 빼면

In [8]:
seen = train[train["attack_cat"] != "Exploits"]
m2 = rf().fit(seen[DESIGNED], seen["label"])
ex = (test["attack_cat"] == "Exploits").values
print(f"평가셋 Exploits {ex.sum()}건 탐지율: 학습에 있을 때 {rf_pred[ex].mean():.2f}"
      f" → 학습에서 뺐을 때 {m2.predict(test[DESIGNED])[ex].mean():.2f}  (비지도 IF {iso_pred[ex].mean():.2f})")

평가셋 Exploits 23건 탐지율: 학습에 있을 때 0.87 → 학습에서 뺐을 때 0.70  (비지도 IF 0.30)


## 5) 다른 시간대: 60분 뒤 구간(later.csv)에서 평가, 모델은 그대로

In [10]:
later = pd.read_csv("../data/processed/later.csv", dtype={"dsport": str})
yl = later["label"]
pl = model.predict_proba(later[DESIGNED])[:, 1]   #predict_proba는 건마다 정상일 확률(0)과 공격일 확률(1) 두 개를 리턴, 여기선 공격 
il = sc.transform(later[DESIGNED])
print(f"{len(later)}건 (공격 {yl.sum()})")
pd.DataFrame({"지도 RF": report(yl, pl, (pl >= 0.5).astype(int)),
              "비지도 IF": report(yl, -iso.score_samples(il), (iso.predict(il) == -1).astype(int))}).T.round(3)

4975건 (공격 233)


,정밀도,재현율,F2,AUC-ROC,오탐,미탐
지도 RF,0.427,0.948,0.762,0.990,297.0,12.0
비지도 IF,0.122,0.266,0.215,0.855,447.0,171.0


### 토론
- 정답으로 학습한 지도 모델이 훨씬 정확합니다. 정답 레이블의 가치입니다.
- 4)에서 처음 보는 Exploits도 일부 잡히는 것은 이 데이터의 공격이 같은 실험망 · 같은 도구에서 나와 모양이 비슷하기 때문입니다. 실무의 새로운 공격은 더 다를 수 있습니다.
- 5)에서 시간대가 바뀌면 정밀도가 내려갑니다 → 주기적 재평가 · 재학습.
- 실무에는 정답이 드뭅니다. 비지도 · 규칙으로 후보를 찾고 → 사람이 판정한 결과를 레이블로 쌓고 → 지도 모델을 키웁니다.